# 04 — Auth / RBAC / Security

بنختبر: التسجيل وتسجيل الدخول، الـ JWT (tampering / expired / alg=none)، الـ refresh rotation والـ logout، **كل route من غير token**، عزل المستخدمين (IDOR)، مصفوفة الصلاحيات Owner/Editor/Viewer، ومدخلات خبيثة أو غريبة (SQLi / XSS / null-byte / طول زيادة).

> أي `FAIL` هنا يعتبر ثغرة أو bug حقيقي، مش مجرد اختلاف في الشكل.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))
sys.path.insert(0, str(pathlib.Path.cwd() / "notebooks"))
from nb_utils import *
env = setup()
print(env)
rec = Recorder("04_auth_rbac_security")

In [ ]:
import uuid as _uuid
api = get_api(); app = api.app
A = api.new_user("owner"); B = api.new_user("outsider")
WS = api.make_workspace(A); PID = api.make_project(A, WS)
print("owner:", A.email, "| workspace:", WS, "| project:", PID)

def probe(token, scheme="Bearer"):
    """Call any protected endpoint with the given token and return the status code."""
    if api.routes.get("me", (0, None))[1]:
        r = api.call("GET", api.routes["me"][1], headers={"Authorization": f"{scheme} {token}"} if token else {})
    else:
        r = api.call("GET", fill(api.routes["workspace_get"][1], WS), headers={"Authorization": f"{scheme} {token}"} if token else {})
    return r.status_code

## 1) Auth basics

In [ ]:
with rec.case("duplicate email is rejected (not 500, not 2xx)", "auth") as c:
    r = api.route("register", json=PAYLOADS["register"](A.email, A.password))
    assert 400 <= r.status_code < 500, f"HTTP {r.status_code}"; c.note(f"HTTP {r.status_code}")

with rec.case("wrong password -> 4xx and no token in the body", "auth") as c:
    r = api.route("login", json=PAYLOADS["login"](A.email, "totally-wrong-password"))
    assert r.status_code in (400, 401, 403, 422), f"HTTP {r.status_code}"
    assert find_key(safe_json(r), "access_token") is None, "token returned for a wrong password!"
    WRONG_PW = r.status_code

with rec.case("unknown email gives the same status as wrong password (no user enumeration)", "auth") as c:
    r = api.route("login", json=PAYLOADS["login"](f"nobody_{_uuid.uuid4().hex[:6]}@example.com", "whatever123"))
    c.note(f"unknown email -> {r.status_code}, wrong password -> {WRONG_PW}")
    if r.status_code != WRONG_PW:
        raise Warn("different status codes reveal which emails are registered")

with rec.case("weak password is rejected", "auth") as c:
    r = api.route("register", json=PAYLOADS["register"](f"weak_{_uuid.uuid4().hex[:6]}@example.com", "123"))
    if r.status_code < 300:
        raise Warn("password '123' was accepted - add a minimum-strength rule")
    c.note(f"HTTP {r.status_code}")

with rec.case("invalid email format is rejected (422)", "auth") as c:
    r = api.route("register", json=PAYLOADS["register"]("not-an-email", "Str0ng!Passw0rd#1"))
    assert r.status_code in (400, 422), f"HTTP {r.status_code}"

with rec.case("password / hash never appear in any response", "auth"):
    r = api.route("register", json=PAYLOADS["register"](f"leak_{_uuid.uuid4().hex[:6]}@example.com", "Str0ng!Passw0rd#9"))
    body = r.text.lower()
    assert "str0ng!passw0rd" not in body and "hashed_password" not in body and "password_hash" not in body and "$2b$" not in body, "credential material leaked in response"

## 2) JWT

In [ ]:
import base64
def b64(d): return base64.urlsafe_b64encode(json.dumps(d).encode()).rstrip(b"=").decode()

with rec.case("valid token is accepted", "jwt"):
    assert probe(A.token) == 200, f"HTTP {probe(A.token)} for a valid token"
with rec.case("no token -> 401/403", "jwt"):
    assert probe(None) in (401, 403)
with rec.case("garbage token -> 401", "jwt"):
    assert probe("not.a.jwt") == 401
with rec.case("tampered signature -> 401", "jwt"):
    t = A.token[:-4] + ("AAAA" if not A.token.endswith("AAAA") else "BBBB"); assert probe(t) == 401
with rec.case("alg=none token (unsigned) -> 401", "jwt"):
    payload = json.loads(base64.urlsafe_b64decode(A.token.split(".")[1] + "=="))
    forged = b64({"alg": "none", "typ": "JWT"}) + "." + b64(payload) + "."
    assert probe(forged) == 401, "server accepted an UNSIGNED token (alg=none) - critical"
with rec.case("wrong Authorization scheme (Basic) -> 401", "jwt"):
    assert probe(A.token, scheme="Basic") in (401, 403)

secret, algo = None, "HS256"
try:
    from helpers import settings as S2
    S2 = S2() if callable(S2) and not hasattr(S2, "model_dump") else S2
    secret = next((str(getattr(S2, k)) for k in ("SECRET_KEY", "JWT_SECRET_KEY", "JWT_SECRET", "SECRET") if hasattr(S2, k)), None)
    algo = next((str(getattr(S2, k)) for k in ("ALGORITHM", "JWT_ALGORITHM") if hasattr(S2, k)), "HS256")
except Exception:
    algo = "HS256"

with rec.case("token signed with a WRONG secret -> 401", "jwt"):
    import jwt as pyjwt
    payload = json.loads(base64.urlsafe_b64decode(A.token.split(".")[1] + "=="))
    assert probe(pyjwt.encode(payload, "wrong-secret-wrong-secret-wrong-secret", algorithm="HS256")) == 401
with rec.case("EXPIRED token (signed with the real secret) -> 401", "jwt") as c:
    import jwt as pyjwt
    if not secret: raise Skip("could not read the JWT secret from settings (SECRET_KEY / JWT_SECRET_KEY ...)")
    payload = json.loads(base64.urlsafe_b64decode(A.token.split(".")[1] + "=="))
    payload["exp"] = int(time.time()) - 3600
    st = probe(pyjwt.encode(payload, secret, algorithm=algo)); assert st == 401, f"expired token accepted, HTTP {st}"
with rec.case("REFRESH token cannot be used as an access token", "jwt"):
    if not A.refresh: raise Skip("login did not return a refresh token")
    assert probe(A.refresh) in (401, 403), "refresh token accepted as access token"

## 3) Refresh rotation و Logout

In [ ]:
U = api.new_user("rot")
with rec.case("refresh returns a NEW token pair", "refresh") as c:
    if not U.refresh: raise Skip("no refresh token issued")
    old_refresh = U.refresh
    r = api.route("refresh", json={"refresh_token": old_refresh})
    assert r.status_code in (200, 201), f"HTTP {r.status_code}: {r.text[:150]}"
    new_access, new_refresh = find_key(safe_json(r), "access_token"), find_key(safe_json(r), "refresh_token")
    assert new_access, "no access_token in refresh response"
    assert new_refresh and new_refresh != old_refresh, "refresh token was NOT rotated"
with rec.case("re-using the OLD refresh token is rejected (rotation / theft detection)", "refresh"):
    need(U.refresh); r = api.route("refresh", json={"refresh_token": old_refresh})
    assert r.status_code in (400, 401, 403), f"old refresh token still works (HTTP {r.status_code})"
with rec.case("logout revokes the refresh token", "refresh") as c:
    need(U.refresh)
    r0 = api.route("login", json=PAYLOADS["login"](U.email, U.password)); tok = find_key(safe_json(r0), "refresh_token"); acc = find_key(safe_json(r0), "access_token")
    r = api.route("logout", token=acc, json={"refresh_token": tok}); assert r.status_code in (200, 204), f"logout HTTP {r.status_code}: {r.text[:150]}"
    r2 = api.route("refresh", json={"refresh_token": tok})
    assert r2.status_code in (400, 401, 403), f"refresh still works after logout (HTTP {r2.status_code})"

## 4) Sweep: كل route من غير token
بنمشي على الـ OpenAPI كله ونندهي كل endpoint محمي بدون Authorization. المتوقع 401/403.

In [ ]:
sweep = []
for path, methods in app.openapi()["paths"].items():
    for m in methods:
        if m.upper() not in ("GET", "POST", "PUT", "PATCH", "DELETE"): continue
        public = bool(re.search(CONFIG["PUBLIC_PATH_REGEX"], path))
        url = re.sub(r"\{[^}]+\}", str(_uuid.uuid4()), path)
        kw = {"json": {}} if m.upper() in ("POST", "PUT", "PATCH") else {}
        r = api.call(m.upper(), url, **kw)
        verdict = "public" if public else ("PASS" if r.status_code in (401, 403) else "FAIL" if (r.status_code < 300 or r.status_code >= 500) else "WARN")
        sweep.append(dict(method=m.upper(), path=path, status=r.status_code, expected="public" if public else "401/403", verdict=verdict))
SW = pd.DataFrame(sweep)
display(SW.sort_values("verdict"))
prot = SW[SW.verdict != "public"]
rec.check("all protected routes reject anonymous calls", (prot.verdict == "PASS").all(),
          f"{int((prot.verdict == 'PASS').sum())}/{len(prot)} OK; FAIL={prot[prot.verdict=='FAIL'].path.tolist()[:6]}; WARN={prot[prot.verdict=='WARN'].path.tolist()[:6]}", "sweep",
          level="fail" if (prot.verdict == "FAIL").any() else "warn")
rec.check("no 5xx from anonymous requests", not (SW.status >= 500).any(), str(SW[SW.status >= 500].path.tolist()), "sweep")

## 5) عزل المستخدمين (IDOR)
Bob (outsider) بيحاول يوصل لحاجات Alice بالـ IDs.

In [ ]:
FID = CID = None
with rec.case("prepare Alice's resources (file + conversation)", "idor") as c:
    r = api.upload(A, PID, "secret_plan.txt", b"Alice private plan: launch code 8841.", "text/plain"); FID = get_id(r)
    CID = api.make_conversation(A, PID); c.note(f"file={FID}", f"conversation={CID}")

def evil_project():
    m, p = api.path("project_create")
    body = PAYLOADS["project"]("evil")
    return api.call(m, fill(p, WS), token=B.token, json=body) if "{" in p else api.call(m, p, token=B.token, json={**body, "workspace_id": WS})

attacks = [
    ("read workspace", lambda: api.route("workspace_get", WS, token=B.token)),
    ("delete workspace", lambda: api.route("workspace_delete", WS, token=B.token)),
    ("read project", lambda: api.route("project_get", PID, token=B.token)),
    ("read file", lambda: api.route("file_get", FID, token=B.token)),
    ("delete file", lambda: api.route("file_delete", FID, token=B.token)),
    ("search inside project", lambda: api.search(B, PID, "launch code")),
    ("read conversation", lambda: api.route("conversation_get", CID, token=B.token)),
    ("post message to conversation", lambda: api.send_message(B, CID, "hello from the outsider")),
    ("create project inside Alice's workspace", lambda: evil_project()),
]
for name, fn in attacks:
    with rec.case(f"outsider cannot: {name}", "idor") as c:
        need(FID, CID)
        r = fn()
        ok = r.status_code in (401, 403, 404) or (r.status_code == 200 and name.startswith("search") and not find_hits(safe_json(r)))
        assert ok, f"HTTP {r.status_code} -> {r.text[:120]}"
        c.note(f"HTTP {r.status_code}")

with rec.case("Alice's data is intact after the attacks", "idor"):
    need(FID)
    assert api.route("file_get", FID, token=A.token).status_code == 200, "file disappeared"
    assert api.route("workspace_get", WS, token=A.token).status_code == 200, "workspace disappeared"

## 6) مصفوفة الصلاحيات RBAC (Owner / Editor / Viewer / Outsider)

الخلية دي بتضيف الأعضاء. **هي الجزء الوحيد اللي ممكن يحتاج تعديل** لأن مسار الدعوة عندك (invite link → join request → approval) مختلف عن إضافة مباشرة. عدّل `add_member()` لو لازم.

In [ ]:
def find_route(method, pattern):
    for p, ops in app.openapi()["paths"].items():
        if method.lower() in ops and re.search(pattern, p):
            return p

def add_member(owner, ws, member, role):
    # ---- ADAPT HERE: wire your invite -> join request -> approve flow if there is no direct "add member" route
    p = find_route("POST", r"/workspaces/\{[^/]+\}/members/?$")
    if p:
        for body in ({"email": member.email, "role": role}, {"user_id": member.id, "role": role}, {"user_id": member.id, "role": role.upper()}):
            r = api.call("POST", fill(p, ws), token=owner.token, json=body)
            if r.status_code in (200, 201):
                return True
    raise Skip("no direct add-member route accepted - wire your invite/join-request flow inside add_member()")

E, V = api.new_user("editor"), api.new_user("viewer")
ROLE_READY = False
with rec.case("add editor + viewer to the workspace", "rbac") as c:
    add_member(A, WS, E, "editor"); add_member(A, WS, V, "viewer"); ROLE_READY = True

actors = {"owner": A, "editor": E, "viewer": V, "outsider": B}
def act_read(u):    return api.route("workspace_get", WS, token=u.token)
def act_project(u):
    m, p = api.path("project_create")
    body = PAYLOADS["project"](f"rbac-{_uuid.uuid4().hex[:5]}")
    return api.call(m, fill(p, WS), token=u.token, json=body) if "{" in p else api.call(m, p, token=u.token, json={**body, "workspace_id": WS})
def act_upload(u):  return api.upload(u, PID, f"rbac_{_uuid.uuid4().hex[:4]}.txt", b"rbac test file", "text/plain")
def act_search(u):  return api.search(u, PID, "launch code")
def act_ask(u):     return api.send_message(u, CID, "Can I ask a question?")
def act_delete_ws(u): return api.route("workspace_delete", WS, token=u.token)

MATRIX = [  # action, function, who is allowed
    ("read workspace", act_read, {"owner", "editor", "viewer"}),
    ("create project", act_project, {"owner", "editor"}),
    ("upload file", act_upload, {"owner", "editor"}),
    ("search", act_search, {"owner", "editor", "viewer"}),
    ("ask in conversation", act_ask, {"owner", "editor", "viewer"}),
    ("delete workspace", act_delete_ws, {"owner"}),   # keep LAST
]
grid = []
for action, fn, allowed in MATRIX:
    order = [r for r in actors if r != "owner"] + ["owner"] if action == "delete workspace" else list(actors)
    for role in order:
        user = actors[role]
        with rec.case(f"{role:8} -> {action}", "rbac") as c:
            need(ROLE_READY or None)
            r = fn(user); ok = r.status_code < 300; should = role in allowed
            grid.append(dict(action=action, role=role, http=r.status_code, allowed=ok, expected=should))
            assert ok == should, f"HTTP {r.status_code}: {'allowed' if ok else 'denied'} but should be {'allowed' if should else 'denied'}"
            if not should: assert r.status_code in (401, 403, 404), f"denied with unexpected HTTP {r.status_code}"
if grid:
    G = pd.DataFrame(grid); G["result"] = G.apply(lambda x: "✅" if x.allowed == x.expected else "❌", axis=1)
    display(G.pivot(index="action", columns="role", values="result").reindex([m[0] for m in MATRIX]))

## 7) مدخلات خبيثة / غريبة
المطلوب: **ولا 500 أبداً**، والبيانات ترجع زي ما اتبعتت (من غير ما تتشوّه أو تتنفذ).

In [ ]:
WS2 = api.make_workspace(B); PID_B = api.make_project(B, WS2)
payloads = {
    "SQL injection": "x'); DROP TABLE users;--",
    "SQL boolean": "' OR '1'='1",
    "XSS script tag": "<script>alert(1)</script>",
    "Arabic + RTL marks": "مساحة عمل \u202e اختبار",
    "emoji + CJK": "🚀 测试 テスト",
    "unicode combining": "e\u0301" * 20,
    "NULL byte": "abc\x00def",
    "very long (10k chars)": "A" * 10_000,
    "only whitespace": "   ",
    "empty string": "",
}
for label, val in payloads.items():
    with rec.case(f"workspace name: {label}", "input-fuzz") as c:
        r = api.route("workspace_create", token=B.token, json=PAYLOADS["workspace"](val))
        assert r.status_code < 500, f"SERVER ERROR HTTP {r.status_code}: {r.text[:150]}"
        if r.status_code in (200, 201):
            back = find_key(unwrap(safe_json(r)), "name")
            if label not in ("whitespace", "only whitespace"):
                assert back == val or (back or "").strip() == val.strip(), f"stored value differs: {str(back)[:40]!r}"
        c.note(f"HTTP {r.status_code}")

for label, q in {"SQL injection": "x' OR 1=1; --", "very long query (20k)": "word " * 4000, "empty query": "", "NULL byte": "a\x00b", "emoji": "🚀🔥"}.items():
    with rec.case(f"search query: {label}", "input-fuzz") as c:
        r = api.search(B, PID_B, q)
        assert r.status_code < 500, f"SERVER ERROR HTTP {r.status_code}: {r.text[:150]}"
        c.note(f"HTTP {r.status_code}")

with rec.case("malformed JSON body -> 422, no stack trace leaked", "input-fuzz"):
    m, p = api.path("workspace_create")
    r = api.call(m, p, token=B.token, content=b"{bad json", headers={"Content-Type": "application/json"})
    assert r.status_code in (400, 422), f"HTTP {r.status_code}"
    assert "traceback" not in r.text.lower() and "File \"" not in r.text, "stack trace leaked to the client"
with rec.case("wrong content-type / body on JSON route does not 500", "input-fuzz"):
    m, p = api.path("workspace_create")
    r = api.call(m, p, token=B.token, content=b"name=abc", headers={"Content-Type": "text/plain"})
    assert r.status_code < 500, f"HTTP {r.status_code}"

## 8) Headers و شكل الـ Errors

In [ ]:
with rec.case("error responses share one consistent JSON shape", "http") as c:
    shapes = {}
    samples = {"401": api.call("GET", fill(api.routes["workspace_get"][1], WS)),
               "404": api.route("workspace_get", str(_uuid.uuid4()), token=A.token),
               "422": api.route("workspace_create", token=A.token, json={})}
    for k, r in samples.items():
        js = safe_json(r); shapes[k] = tuple(sorted(js.keys())) if isinstance(js, dict) else type(js).__name__
    c.note(shapes)
    if len(set(shapes.values())) > 1:
        raise Warn(f"error bodies have different shapes: {shapes} (frontend toast needs one predictable field)")

with rec.case("CORS preflight does not allow every origin together with credentials", "http") as c:
    r = api.call("OPTIONS", api.routes["workspace_create"][1], headers={"Origin": "http://evil.example", "Access-Control-Request-Method": "POST"})
    aco, acc = r.headers.get("access-control-allow-origin"), r.headers.get("access-control-allow-credentials")
    c.note(f"allow-origin={aco}", f"allow-credentials={acc}")
    if aco == "*" and str(acc).lower() == "true": raise AssertionError("wildcard origin with credentials")
    if aco in ("*", "http://evil.example"): raise Warn("any origin is allowed - restrict CORS origins in production")

with rec.case("security headers present (informational)", "http") as c:
    r = api.call("GET", "/docs")
    want = ["x-content-type-options", "x-frame-options", "strict-transport-security", "content-security-policy"]
    missing = [h for h in want if h not in {k.lower() for k in r.headers}]
    if missing: raise Warn(f"missing: {missing} (usually added by the reverse proxy / nginx)")

In [ ]:
rec.summary(); rec.table(); rec.save()